# 085 — Cuantización e inferencia local

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

- **Cuantización afín**: x ≈ s·(q − z); simétrica para pesos: s = max|x|/127 en
  INT8. Error acotado por s/2 → escalas por grupo/bloque para que un outlier no
  arruine al resto.
- **Outliers**: desde ~6,7B aparecen dimensiones de activación atípicas;
  LLM.int8() las computa en FP16 (descomposición mixta); GPTQ y AWQ cuantizan solo
  pesos minimizando el error de salida o protegiendo canales salientes.
- **Escalera de bits**: INT8 ≈ gratis; 4 bits bien hecho pierde poco; 2 bits
  colapsa. PTQ (con calibración) es el estándar; QAT exige reentrenar.
- **GGUF/llama.cpp**: un archivo con pesos cuantizados por bloques (Q4_K_M ≈ 4,55
  bits/peso) + tokenizador; un 8B en Q4 ≈ 4,9 GB corre en CPU de portátil —
  privacidad y costo marginal cero a cambio de calidad/velocidad.
- **Por qué acelera**: el decode está limitado por bytes movidos, no por FLOPs.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("neural", seed=85)
show(result)


## Reflexión

1. ¿Por qué la cuantización acelera el decode aunque la GPU tenga que descuantizar
   cada peso antes de multiplicar?
2. Un outlier de magnitud 12 entre pesos de magnitud <1 multiplica el error de
   todos: ¿qué dos técnicas distintas de la clase lo neutralizan y en qué difieren?
3. ¿En qué escenarios el modelo local en Q4 supera *en la práctica* a un modelo
   mayor vía API, aun perdiendo en benchmarks de calidad?